
# Incident Analysis With LLM

Ноутбук делает три шага:
1. Загружает таблицу в формате `Incident_analysis_march.ipynb`.
2. Выполняет минимальную предобработку через `IncidentRequestsPreprocessor` без `natasha`.
3. Классифицирует каждую заявку с помощью LLM и добавляет колонку `Тип инцидента`.

Важные точки настройки находятся в ячейке `Конфиг`: путь к данным, список `COLS2DROP`, список `INCIDENT_TYPES` и `MODEL_NAME`.


In [1]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd()
if not (ROOT / "incident_requests").exists() and (ROOT.parent / "incident_requests").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)


/home/fedor/Projects/building_maintenance_agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Конфиг
INPUT_PATH = ROOT / "research/data" / "march_incidents.xlsx"
SHEET_NAME = 0

COLS2DROP = [
    "Источник",
    "Категория",
]

TEXT_COLUMN = "Описание"
TYPE_COLUMN = "Тип инцидента"
OTHER_LABEL = "Прочее"
EMPTY_LABEL = "Не определен"

INCIDENT_TYPES = [
    "Стояк ГВС",
    "Труба ГВС",
    "Стояк ХВС",
    "Труба ХВС",
]

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 96
OUTPUT_PATH = ROOT / "research" / "incident_analysis_llm_output.parquet"


In [3]:
def load_table(path: Path, sheet_name=0) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)

    if suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Неподдерживаемый формат файла: {suffix}")


df = load_table(INPUT_PATH, sheet_name=SHEET_NAME)
processor = IncidentRequestsPreprocessor(
    df,
    columns_to_drop=COLS2DROP,
    detect_incident_type=False,
    use_natasha=False,
)
preprocessed_df = processor.preprocess()

print(preprocessed_df.dtypes)
display(preprocessed_df.head())


FileNotFoundError: [Errno 2] No such file or directory: '/home/fedor/Projects/building_maintenance_agents/data/march_incidents.xlsx'

In [ ]:

def build_classification_prompt(
    description: str,
    incident_types: list[str],
    other_label: str = OTHER_LABEL,
) -> str:
    labels = [*incident_types, other_label]
    options = "\n".join(f"- {label}" for label in labels)

    return f"""Определи тип инцидента по тексту заявки.
Выбери ровно один вариант из списка.
Если ни один вариант не подходит, выбери "{other_label}".

Доступные типы инцидентов:
{options}

Текст заявки:
{description}

Верни только JSON без пояснений:
{{"incident_type": "<один вариант из списка>"}}"""


def load_generation_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()

    return tokenizer, model


In [ ]:

def parse_model_response(response_text: str, labels: list[str], other_label: str) -> str:
    match = re.search(r"\{.*\}", response_text, flags=re.S)
    if match:
        try:
            payload = json.loads(match.group(0))
            label = str(payload.get("incident_type", "")).strip()
            if label in labels:
                return label
        except json.JSONDecodeError:
            pass

    cleaned_text = response_text.strip().strip('"')
    if cleaned_text in labels:
        return cleaned_text

    for label in labels:
        if label.lower() in cleaned_text.lower():
            return label

    return other_label


def generate_incident_type(
    description: str,
    tokenizer,
    model,
    incident_types: list[str],
    other_label: str = OTHER_LABEL,
    empty_label: str = EMPTY_LABEL,
) -> str:
    if pd.isna(description) or not str(description).strip():
        return empty_label

    labels = [*incident_types, other_label]
    messages = [
        {
            "role": "system",
            "content": "Ты размечаешь заявки ЖКХ и возвращаешь только валидный JSON.",
        },
        {
            "role": "user",
            "content": build_classification_prompt(str(description), incident_types, other_label),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = generated[0][model_inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return parse_model_response(response_text, labels, other_label)


def classify_incidents(
    dataframe: pd.DataFrame,
    tokenizer,
    model,
    text_column: str,
    incident_types: list[str],
) -> pd.DataFrame:
    if text_column not in dataframe.columns:
        raise ValueError(f"Колонка '{text_column}' не найдена. Есть: {list(dataframe.columns)}")

    result_df = dataframe.copy()
    result_df[TYPE_COLUMN] = [
        generate_incident_type(description, tokenizer, model, incident_types)
        for description in tqdm(result_df[text_column], total=len(result_df))
    ]
    return result_df


In [ ]:

tokenizer, model = load_generation_model(MODEL_NAME)
result_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=TEXT_COLUMN,
    incident_types=INCIDENT_TYPES,
)

display(result_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    result_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)


In [ ]:

if OUTPUT_PATH.suffix == ".csv":
    result_df.to_csv(OUTPUT_PATH, index=False)
elif OUTPUT_PATH.suffix in {".xlsx", ".xls"}:
    result_df.to_excel(OUTPUT_PATH, index=False)
else:
    result_df.to_parquet(OUTPUT_PATH, index=False)

OUTPUT_PATH
